# Inventario de datos de Olist (E-Commerce)

Este notebook arma un mapa general del dataset de Olist antes de entrar en
cualquier mision especifica. No responde ninguna pregunta de negocio todavia:
documenta que tablas hay, que tamano tienen, como se conectan entre si y donde
hay valores nulos. Cada mision (recomendaciones, entregas, sentimiento) parte de
este conocimiento base y hace su propio analisis exploratorio enfocado en su
propia pregunta.

In [1]:
# pandas: manipular tablas y leer CSV
import pandas as pd
# Path: rutas de archivos con separadores correctos
from pathlib import Path

# data_raw: ruta a la carpeta ../data/raw (relativa a notebooks/)
data_raw = Path("..") / "data" / "raw"

In [2]:
# Mapa: nombre simbolico de cada tabla -> nombre real del CSV
archivos = {
    "customers": "olist_customers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}

# Carga todas las tablas en un dict nombre -> DataFrame
tablas = {nombre: pd.read_csv(data_raw / archivo) for nombre, archivo in archivos.items()}

## Recuento de tablas y su tamano

Cuantas filas y columnas tiene cada tabla. Esto da una primera nocion de la escala
de cada una: por ejemplo, order_items deberia tener mas filas que orders, porque un
pedido puede tener varios items.

In [3]:
# shape: tupla (filas, columnas) de cada tabla
shapes = {nombre: df.shape for nombre, df in tablas.items()}
# Transforma el dict a DataFrame para leerlo en filas (una por tabla)
pd.DataFrame(shapes, index=["filas", "columnas"]).T

,filas,columnas
customers,99441,5
geolocation,1000163,5
orders,99441,8
order_items,112650,7
order_payments,103886,5
order_reviews,99224,7
products,32951,9
sellers,3095,4
category_translation,71,2


## Mapa de relaciones entre tablas

El dataset esta normalizado: cada tabla cubre una entidad, y se conectan entre si
por claves compartidas.

| Tabla A | Clave | Tabla B |
|---|---|---|
| orders | customer_id | customers |
| orders | order_id | order_items |
| order_items | product_id | products |
| order_items | seller_id | sellers |
| orders | order_id | order_payments |
| orders | order_id | order_reviews |
| products | product_category_name | category_translation |
| customers | customer_zip_code_prefix | geolocation (zip_code_prefix) |
| sellers | seller_zip_code_prefix | geolocation (zip_code_prefix) |

orders es la tabla central: casi todo el resto se conecta a ella directa o
indirectamente a traves de order_id o customer_id.

## Perfil de tipos y valores faltantes

Para cada tabla: cuantas columnas hay de cada tipo de dato, y que porcentaje de
valores nulos tiene cada columna. Esto anticipa trabajo de limpieza que cada
mision va a tener que resolver por su cuenta.

In [4]:
# Recorre cada tabla del inventario
for nombre, df in tablas.items():
# Construye un resumen por columna
    resumen = pd.DataFrame({
# dtype: tipo de dato de la columna
        "dtype": df.dtypes,
# pct_nulos: proporcion de nulos convertida a porcentaje
        "pct_nulos": (df.isna().mean() * 100).round(2),
    })
# Imprime el encabezado con el nombre de la tabla
    print(f"--- {nombre} ---")
# Imprime el resumen de tipos y nulos
    print(resumen)
# Deja una linea en blanco entre tablas
    print()

--- customers ---
                          dtype  pct_nulos
customer_id                 str        0.0
customer_unique_id          str        0.0
customer_zip_code_prefix  int64        0.0
customer_city               str        0.0
customer_state              str        0.0

--- geolocation ---
                               dtype  pct_nulos
geolocation_zip_code_prefix    int64        0.0
geolocation_lat              float64        0.0
geolocation_lng              float64        0.0
geolocation_city                 str        0.0
geolocation_state                str        0.0

--- orders ---
                              dtype  pct_nulos
order_id                        str       0.00
customer_id                     str       0.00
order_status                    str       0.00
order_purchase_timestamp        str       0.00
order_approved_at               str       0.16
order_delivered_carrier_date    str       1.79
order_delivered_customer_date   str       2.98
order_estimated_deliver